In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import sklearn.metrics
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
from torch.optim import Adam
import torch.utils.data as Data
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from collections import Counter
import copy
import random
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import seaborn as sns 

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    
    print(f" Random seed set: {seed}")

set_seed(42)


In [ ]:
'''
# 'multi' : RNA + ADT
MODALITY = 'multi'



DATA_FORMAT = 'mtx'
DATASET_ID  = 7

MTX_DATASETS = {
    7: (
        "Experiments/7_GSE194122_s3d6/",
        "GSE194122_s3d6_matrix.mtx",
        "GSE194122_s3d6_features.tsv",
        "GSE194122_s3d6_barcodes.tsv",
        "GSE194122_s3d6_ADT.csv",
        "GSE194122_s3d6_label.csv",
        None
    ),
}

H5AD_DATASETS = {
    1: (
        "Experiments/1_10X1kpbmc/",
        "10X1kpbmc_rna.h5ad",
        "10X1kpbmc_adt.h5ad",
        "10X1kpbmc_label.csv"
    ),
   
}


print("\n" + "="*70)
print("Step 1: Loading multimodal data (RNA + ADT)")
print("="*70)

if DATA_FORMAT == 'mtx':
    cfg            = MTX_DATASETS[DATASET_ID]
    data_dir       = cfg[0]
    rna_mtx        = cfg[1]
    rna_feat_file  = cfg[2]
    rna_bc_file    = cfg[3]
    adt_csv        = cfg[4]
    label_csv      = cfg[5]
    label_replace  = cfg[6]   # None or ('-', '.')

    print("\n[1.1] Loading RNA expression data (MTX format)...")
    adata_rna = sc.read_mtx(data_dir + rna_mtx).T
    features  = pd.read_csv(data_dir + rna_feat_file, sep='\t', header=None)
    barcodes  = pd.read_csv(data_dir + rna_bc_file,   sep='\t', header=None)
    adata_rna.var_names = features[0].values
    adata_rna.obs_names = barcodes[0].values
    print(f"  RNA data: {adata_rna.shape} (cells × genes)")

    print("\n[1.2] Loading ADT protein expression data (CSV format)...")
    adt_data = pd.read_csv(data_dir + adt_csv, index_col=0).T
    if DATASET_ID == 14:
        adt_data.index = adt_data.index.str.replace('.', '-', regex=False)
    print(f"  ADT data: {adt_data.shape} (cells × proteins)")

    print("\n[1.3] Loading cell-type labels...")
    label = pd.read_csv(data_dir + label_csv, index_col=0)
    if label_replace is not None:
        old, new = label_replace
        label.index = label.index.str.replace(old, new, regex=False)
    print(f"  Label data: {label.shape}")

elif DATA_FORMAT == 'h5ad':
    cfg            = H5AD_DATASETS[DATASET_ID]
    data_dir       = cfg[0]
    rna_h5ad_file  = cfg[1]
    adt_h5ad_file  = cfg[2]
    label_csv      = cfg[3]

    print("\n[1.1] Loading RNA expression data (H5AD format)...")
    adata_rna = sc.read_h5ad(data_dir + rna_h5ad_file)
    adata_rna.obs_names = adata_rna.obs_names.astype(str)
    print(f"  RNA data: {adata_rna.shape} (cells × genes)")
    print(f"  RNA obs columns: {list(adata_rna.obs.columns)}")
    print(f"  RNA var columns: {list(adata_rna.var.columns)}")

    print("\n[1.2] Loading ADT protein expression data (H5AD format)...")
    adata_adt = sc.read_h5ad(data_dir + adt_h5ad_file)
    adata_adt.obs_names = adata_adt.obs_names.astype(str)
    print(f"  ADT data: {adata_adt.shape} (cells × proteins)")
    print(f"  ADT obs columns: {list(adata_adt.obs.columns)}")
    print(f"  ADT var columns: {list(adata_adt.var.columns)}")

    if sp.issparse(adata_adt.X):
        adt_matrix = adata_adt.X.toarray()
    else:
        adt_matrix = np.array(adata_adt.X)

    adt_data = pd.DataFrame(
        adt_matrix,
        index=adata_adt.obs_names,
        columns=adata_adt.var_names
    )
    print(f"  ADT DataFrame: {adt_data.shape}")

    print("\n[1.3] Loading cell-type labels (CSV format)...")
    label = pd.read_csv(data_dir + label_csv, index_col=0)
    label.index = label.index.astype(str)
    print(f"  Label data: {label.shape}")

else:
    raise ValueError(f"Unsupported data format: {DATA_FORMAT}, please choose 'mtx' or 'h5ad'")

print(f"\n  Cell types: {label.iloc[:, 0].unique()}")
print(f"  Counts per cell type:\n{label.iloc[:, 0].value_counts()}")

# -------------------- Initial alignment --------------------
common_cells = (adata_rna.obs_names
                .intersection(adt_data.index)
                .intersection(label.index))
print(f"\n[1.4] Initial alignment: {len(common_cells)} common cells")

if len(common_cells) == 0:
    print(" Alignment failed. Please check the index format:")
    print(f"  RNA  obs_names first 5: {list(adata_rna.obs_names[:5])}")
    print(f"  ADT  index     first 5: {list(adt_data.index[:5])}")
    print(f"  Label index    first 5: {list(label.index[:5])}")
    raise ValueError("No common cells were found among the three datasets. Please check whether barcode formats are consistent.")

adata_rna = adata_rna[common_cells, :].copy()
adt_data  = adt_data.loc[common_cells, :].copy()
label     = label.loc[common_cells, :].copy()


# ==================== Step 2: RNA data preprocessing ====================
print("\n" + "="*70)
print("Step 2: RNA data preprocessing")
print("="*70)

print(f"Before preprocessing: {adata_rna.n_obs} cells")

sc.pp.filter_cells(adata_rna, min_genes=200)
sc.pp.filter_genes(adata_rna, min_cells=3)
print(f"After filtering: {adata_rna.n_obs} cells(removed {len(common_cells) - adata_rna.n_obs} low-quality cells)")

sc.pp.normalize_total(adata_rna, target_sum=1e4)
sc.pp.log1p(adata_rna)

sc.pp.highly_variable_genes(adata_rna, n_top_genes=2000, subset=True)
print(f"Feature selection: retained {adata_rna.n_vars} highly variable genes")

filtered_cells = adata_rna.obs_names
print(f"\n✓ RNA preprocessing completed: {len(filtered_cells)} cells × {adata_rna.n_vars} genes")


# ==================== Step 3: Aligning ADT and labels based on filtered RNA cells ====================
print("\n" + "="*70)
print("Step 3: Aligning ADT and labels based on filtered RNA cells")
print("="*70)

adt_aligned   = adt_data.loc[filtered_cells].copy()
label_aligned = label.loc[filtered_cells].copy()

print(f"ADT after alignment: {adt_aligned.shape}")
print(f"Label after alignment: {label_aligned.shape}")

print("\n[3.1] ADT data normalization...")
adt_array      = adt_aligned.values.astype(np.float64)
adt_array      = np.log1p(adt_array)
scaler         = StandardScaler()
adt_normalized = scaler.fit_transform(adt_array)
print(f"  ✓ ADT normalization completed: {adt_normalized.shape}")

assert len(adata_rna) == len(adt_aligned) == len(label_aligned), "Data alignment failed."
print(f"\n✓ Data alignment check passed: {len(filtered_cells)} cellsfully aligned")


# ==================== Step 4: Modality selection / ablation experiment ====================
print("\n" + "=" * 70)
print("Step 4: Modality selection / ablation experiment")
print("=" * 70)

rna_array = adata_rna.X.toarray() if sp.issparse(adata_rna.X) else np.array(adata_rna.X)
print(f"RNA data shape: {rna_array.shape}")
print(f"ADT data shape: {adt_normalized.shape}")

if MODALITY == 'multi':
    # RNA + ADT
    fused_array = np.hstack([rna_array, adt_normalized])
    rna_columns = [f"RNA_{i}" for i in range(rna_array.shape[1])]
    adt_columns = [f"ADT_{i}" for i in range(adt_normalized.shape[1])]
    all_columns = rna_columns + adt_columns

    print("\n✓ Current mode: RNA + ADT multimodal")
    print(f"  - RNA features: {rna_array.shape[1]} dimensions")
    print(f"  - ADT features: {adt_normalized.shape[1]} dimensions")
    print(f"  - Fused features: {fused_array.shape[1]} dimensions")

elif MODALITY == 'rna':
    fused_array = rna_array
    all_columns = [f"RNA_{i}" for i in range(rna_array.shape[1])]

    print("\n✓ Current mode: RNA unimodal")
    print(f"  - RNA features: {rna_array.shape[1]} dimensions")

elif MODALITY == 'adt':
    fused_array = adt_normalized
    all_columns = [f"ADT_{i}" for i in range(adt_normalized.shape[1])]

    print("\n✓ Current mode: ADT unimodal")
    print(f"  - ADT features: {adt_normalized.shape[1]} dimensions")

else:
    raise ValueError("MODALITY must be 'multi', 'rna' or 'adt'")

data = pd.DataFrame(
    fused_array,
    index=filtered_cells,
    columns=all_columns
)

print(f"\nFinal data:")
print(f"  Feature matrix: {data.shape}")
print(f"  Labels:     {label_aligned.shape}")
'''

In [ ]:

def geneSelection(data, threshold=0, atleast=10,
                 yoffset=0.02, xoffset=5, decay=1.5, n=None,
                 plot=True, markers=None, genes=None, figsize=(6, 3.5),
                 markeroffsets=None, labelsize=10, alpha=1, verbose=1):
    if sp.issparse(data):
        zeroRate = 1 - np.squeeze(np.array((data > threshold).mean(axis=0)))
        A = data.multiply(data > threshold)
        A.data = np.log2(A.data)
        meanExpr = np.zeros_like(zeroRate) * np.nan
        detected = zeroRate < 1
        meanExpr[detected] = np.squeeze(np.array(A[:, detected].mean(axis=0))) / (1 - zeroRate[detected])
    else:
        zeroRate = 1 - np.mean(data > threshold, axis=0)
        meanExpr = np.zeros_like(zeroRate) * np.nan
        detected = zeroRate < 1
        mask = data[:, detected] > threshold
        logs = np.zeros_like(data[:, detected]) * np.nan
        logs[mask] = np.log2(data[:, detected][mask])
        meanExpr[detected] = np.nanmean(logs, axis=0)
    
    lowDetection = np.array(np.sum(data > threshold, axis=0)).squeeze() < atleast
    zeroRate[lowDetection] = np.nan
    meanExpr[lowDetection] = np.nan
    
    if n is not None:
        up = 10
        low = 0
        for t in range(100):
            nonan = ~np.isnan(zeroRate)
            selected = np.zeros_like(zeroRate).astype(bool)
            selected[nonan] = zeroRate[nonan] > np.exp(-decay * (meanExpr[nonan] - xoffset)) + yoffset
            if np.sum(selected) == n:
                break
            elif np.sum(selected) < n:
                up = xoffset
                xoffset = (xoffset + low) / 2
            else:
                low = xoffset
                xoffset = (xoffset + up) / 2
        if verbose > 0:
            print(f'Chosen offset: {xoffset:.2f}')
    else:
        nonan = ~np.isnan(zeroRate)
        selected = np.zeros_like(zeroRate).astype(bool)
        selected[nonan] = zeroRate[nonan] > np.exp(-decay * (meanExpr[nonan] - xoffset)) + yoffset
    
    if verbose > 0:
        print(f'{np.sum(selected)} genes selected\n')
    
    if plot:
        if figsize is not None:
            plt.figure(figsize=figsize)
        plt.ylim([0, 1])
        if threshold > 0:
            plt.xlim([np.log2(threshold), np.ceil(np.nanmax(meanExpr))])
        else:
            plt.xlim([0, np.ceil(np.nanmax(meanExpr))])
        x = np.arange(plt.xlim()[0], plt.xlim()[1] + 0.1, 0.1)
        y = np.exp(-decay * (x - xoffset)) + yoffset
        plt.text(0.4, 0.2, '{} genes selected\ny = exp(-{:.1f}*(x-{:.2f}))+{:.2f}'.format(
            np.sum(selected), decay, xoffset, yoffset),
            color='k', fontsize=labelsize, transform=plt.gca().transAxes)
        plt.plot(x, y, color=sns.color_palette()[1], linewidth=2)
        xy = np.concatenate((np.concatenate((x[:, None], y[:, None]), axis=1),
                           np.array([[plt.xlim()[1], 1]])))
        t = plt.matplotlib.patches.Polygon(xy, color=sns.color_palette()[1], alpha=0.4)
        plt.gca().add_patch(t)
        plt.scatter(meanExpr, zeroRate, s=1, alpha=alpha, rasterized=True)
        if threshold == 0:
            plt.xlabel('Mean log2 nonzero expression')
            plt.ylabel('Frequency of zero expression')
        else:
            plt.xlabel('Mean log2 nonzero expression')
            plt.ylabel('Frequency of near-zero expression')
        plt.tight_layout()
        plt.savefig('select.svg')
    
    return selected

MODALITY = 'multi'

DATA_FORMAT = 'mtx'
DATASET_ID = 22

MTX_DATASETS = {
    22: (
        "Experiments/22_GSE126074/",
        "brain_SNARE_RNA_matrix.mtx", "brain_SNARE_RNA_features.tsv", "brain_SNARE_barcodes.tsv",
        "brain_SNARE_ATAC_matrix.mtx", "brain_SNARE_ATAC_features.tsv", "brain_SNARE_barcodes.tsv",
        "brain_SNARE_label.csv"
    )
}

H5AD_DATASETS = {
    19: (
        "Experiments/19_human_brain_3k/",
        "human_brain_3k_rna.h5ad",
        "human_brain_3k_atac.h5ad",
        "human_brain_3k_label.csv"
    )
}




print("\n" + "="*70)
print("Step 1: Loading multimodal data (RNA + ATAC)")
print("="*70)
print(f"  Data format: {DATA_FORMAT.upper()}")
print(f"  Dataset ID: {DATASET_ID}")

if DATA_FORMAT == 'mtx':
    cfg = MTX_DATASETS[DATASET_ID]
    data_dir        = cfg[0]
    rna_mtx         = cfg[1]
    rna_feat_file   = cfg[2]
    rna_bc_file     = cfg[3]
    atac_mtx        = cfg[4]
    atac_feat_file  = cfg[5]
    atac_bc_file    = cfg[6]
    label_file      = cfg[7]

    print("\n[1.1] Loading RNA expression data (MTX format)...")
    adata_rna = sc.read_mtx(data_dir + rna_mtx).T
    rna_features = pd.read_csv(data_dir + rna_feat_file, sep='\t', header=None)
    rna_barcodes = pd.read_csv(data_dir + rna_bc_file,   sep='\t', header=None)
    adata_rna.var_names = rna_features[0].values
    adata_rna.obs_names = rna_barcodes[0].values
    print(f"  RNA data: {adata_rna.shape} (cells × genes)")

    print("\n[1.2] Loading ATAC chromatin accessibility data (MTX format)...")
    adata_atac = sc.read_mtx(data_dir + atac_mtx).T
    atac_features = pd.read_csv(data_dir + atac_feat_file, sep='\t', header=None)
    atac_barcodes = pd.read_csv(data_dir + atac_bc_file,   sep='\t', header=None)
    adata_atac.var_names = atac_features[0].values
    adata_atac.obs_names = atac_barcodes[0].values
    print(f"  ATAC data: {adata_atac.shape} (cells × peak regions)")

    print("\n[1.3] Loading cell-type labels...")
    label = pd.read_csv(data_dir + label_file, index_col=0)

elif DATA_FORMAT == 'h5ad':
    cfg = H5AD_DATASETS[DATASET_ID]
    data_dir       = cfg[0]
    rna_h5ad_file  = cfg[1]
    atac_h5ad_file = cfg[2]
    label_file     = cfg[3]

    print("\n[1.1] Loading RNA expression data (H5AD format)...")
    adata_rna = sc.read_h5ad(data_dir + rna_h5ad_file)
    adata_rna.obs_names = adata_rna.obs_names.astype(str)
    print(f"  RNA data: {adata_rna.shape} (cells × genes)")
    print(f"  RNA obs columns: {list(adata_rna.obs.columns)}")
    print(f"  RNA var columns: {list(adata_rna.var.columns)}")

    print("\n[1.2] Loading ATAC chromatin accessibility data (H5AD format)...")
    adata_atac = sc.read_h5ad(data_dir + atac_h5ad_file)
    adata_atac.obs_names = adata_atac.obs_names.astype(str)
    print(f"  ATAC data: {adata_atac.shape} (cells × peak regions)")
    print(f"  ATAC obs columns: {list(adata_atac.obs.columns)}")
    print(f"  ATAC var columns: {list(adata_atac.var.columns)}")

    print("\n[1.3] Loading cell-type labels...")
    label = pd.read_csv(data_dir + label_file, index_col=0)
    label.index = label.index.astype(str)

else:
    raise ValueError(f"Unsupported data format: {DATA_FORMAT}, please choose 'mtx' or 'h5ad'")

print(f"  Label data: {label.shape}")
print(f"  Cell types: {label.iloc[:, 0].unique()}")
print(f"  Counts per cell type:\n{label.iloc[:, 0].value_counts()}")

# ==================== Initial alignment ====================
common_cells = adata_rna.obs_names.intersection(
                   adata_atac.obs_names).intersection(label.index)
print(f"\n[1.4] Initial alignment: {len(common_cells)} common cells")

if len(common_cells) == 0:
    print(" Alignment failed. Please check the index format:")
    print(f"  RNA  obs_names first 5: {list(adata_rna.obs_names[:5])}")
    print(f"  ATAC obs_names first 5: {list(adata_atac.obs_names[:5])}")
    print(f"  Label index    first 5: {list(label.index[:5])}")
    raise ValueError("No common cells were found among the three datasets. Please check whether barcode formats are consistent.")

adata_rna  = adata_rna[common_cells, :].copy()
adata_atac = adata_atac[common_cells, :].copy()
label      = label.loc[common_cells, :].copy()


# ==================== Step 2: RNA data preprocessing ====================
print("\n" + "="*70)
print("Step 2: RNA data preprocessing")
print("="*70)

print(f"Before preprocessing: {adata_rna.n_obs} cells")

sc.pp.filter_cells(adata_rna, min_genes=200)
sc.pp.filter_genes(adata_rna, min_cells=3)
print(f"After filtering: {adata_rna.n_obs} cells(removed {len(common_cells) - adata_rna.n_obs} low-quality cells)")

sc.pp.normalize_total(adata_rna, target_sum=1e4)
sc.pp.log1p(adata_rna)

sc.pp.highly_variable_genes(adata_rna, n_top_genes=2000, subset=True)
print(f"Feature selection: retained {adata_rna.n_vars} highly variable genes")

filtered_cells = adata_rna.obs_names
print(f"\n✓ RNA preprocessing completed: {len(filtered_cells)} cells × {adata_rna.n_vars} genes")


# ==================== Step 3: ATAC data preprocessing (feature selection) ====================
print("\n" + "="*70)
print("Step 3: ATAC data preprocessing (feature selection)")
print("="*70)

adata_atac_aligned = adata_atac[filtered_cells, :].copy()
print(f"ATAC after alignment: {adata_atac_aligned.shape}")

if sp.issparse(adata_atac_aligned.X):
    atac_array = adata_atac_aligned.X.toarray()
else:
    atac_array = np.array(adata_atac_aligned.X)

print(f"\n[3.1] Running geneSelection for ATAC feature selection...")
print(f"  Input: {atac_array.shape} (cells × peak regions)")

selected_features = geneSelection(
    atac_array,
    threshold=0,
    atleast=10,
    yoffset=0.02,
    xoffset=5,
    decay=1.5,
    n=2000,
    plot=True,
    markers=None,
    genes=None,
    figsize=(6, 3.5),
    verbose=1
)

atac_selected = atac_array[:, selected_features]
print(f"  Output: {atac_selected.shape} (retained {np.sum(selected_features)} highly variable peaks)")

print("\n[3.2] ATAC data normalization...")
from sklearn.preprocessing import normalize

atac_log = np.log1p(atac_selected)
n_cells  = atac_log.shape[0]
idf      = np.log(n_cells / (1 + np.sum(atac_selected > 0, axis=0)))
atac_tfidf      = atac_log * idf
atac_normalized = normalize(atac_tfidf, norm='l2', axis=1)
print(f"  ✓ ATAC normalization completed: {atac_normalized.shape}")


# ==================== Step 4: Aligning labels based on filtered RNA cells ====================
print("\n" + "="*70)
print("Step 4: Aligning labels based on filtered RNA cells")
print("="*70)

label_aligned = label.loc[filtered_cells].copy()
print(f"Label after alignment: {label_aligned.shape}")

assert len(adata_rna) == len(atac_normalized) == len(label_aligned), "Data alignment failed."
print(f"\n✓ Data alignment check passed: {len(filtered_cells)} cellsfully aligned")


# ==================== Step 5: Modality selection ====================
print("\n" + "="*70)
print("Step 5: Modality selection")
print("="*70)

rna_array = adata_rna.X.toarray() if sp.issparse(adata_rna.X) else np.array(adata_rna.X)

print(f"RNA data shape:  {rna_array.shape}")
print(f"ATAC data shape: {atac_normalized.shape}")

# ---------- multimodal ----------
if MODALITY == 'multi':
    fused_array = np.hstack([rna_array, atac_normalized])

    rna_columns  = [f"RNA_{i}" for i in range(rna_array.shape[1])]
    atac_columns = [f"ATAC_{i}" for i in range(atac_normalized.shape[1])]
    all_columns  = rna_columns + atac_columns

    print("\n✓ Using RNA + ATAC multimodal data")

# ---------- RNA-only ----------
elif MODALITY == 'rna':
    fused_array = rna_array

    all_columns = [f"RNA_{i}" for i in range(rna_array.shape[1])]

    print("\n✓ Using RNA unimodal data")

# ---------- ATAC-only ----------
elif MODALITY == 'atac':
    fused_array = atac_normalized

    all_columns = [f"ATAC_{i}" for i in range(atac_normalized.shape[1])]

    print("\n✓ Using ATAC unimodal data")

else:
    raise ValueError("MODALITY must be 'multi', 'rna' or 'atac'")

print(f"\nFinal feature shape: {fused_array.shape}")

data = pd.DataFrame(
    fused_array,
    index=filtered_cells,
    columns=all_columns
)

print(f"\nFinal data:")
print(f"  Feature matrix: {data.shape}")
print(f"  Labels:     {label_aligned.shape}")




In [ ]:
print("\n" + "="*70)
print("Step 6: Dataset splitting (normal cells vs rare cells)")
print("="*70)

def build_dataset(data, label, drop_number=1):
    """
    5. Final test set = 20%normal cells + 100%rare cells
    """
    label_col = label.columns[0]
    counts    = label[label_col].value_counts()

    print(f"\nCounts per cell type:")
    print(counts.to_string())

    rare_types = counts.nsmallest(drop_number).index.tolist()
    print(f"\nRare Cell types: {rare_types}")
    print(f"Rare cell counts: {[counts[rt] for rt in rare_types]}")

    is_rare = label[label_col].isin(rare_types)
    X_ood   = data.loc[is_rare]
    y_ood   = label.loc[is_rare]
    X_id    = data.loc[~is_rare]
    y_id    = label.loc[~is_rare]

    print(f"\nNormal cells: {len(X_id)} ")
    print(f"Rare cells: {len(X_ood)} ")

    while True:
        id_counts   = y_id[label_col].value_counts()
        problematic = id_counts[id_counts < 5].index.tolist()

        if len(problematic) == 0:
            break

        print(f"\nThe following classes have fewer than 5 samples and are moved to the rare-cell set: {problematic}")

        is_problematic = y_id[label_col].isin(problematic)
        X_ood = pd.concat([X_ood, X_id.loc[is_problematic]])
        y_ood = pd.concat([y_ood, y_id.loc[is_problematic]])
        X_id  = X_id.loc[~is_problematic]
        y_id  = y_id.loc[~is_problematic]

        print(f"After adjustment -> Normal cells: {len(X_id)} , Rare cells: {len(X_ood)} ")

    X_train, X_test_id, y_train, y_test_id = train_test_split(
        X_id, y_id,
        test_size=0.2,
        random_state=42,
        stratify=y_id
    )

    print(f"\n Dataset splitting completed:")
    print(f"  Training set: {len(X_train)} normal cells")
    print(f"  Test set: {len(X_test_id)} normal cells + {len(X_ood)} rare cells")

    return X_train, X_test_id, y_train, y_test_id, X_ood, y_ood


X_train, X_test_id, y_train, y_test_id, X_ood, y_ood = build_dataset(
    data, label_aligned, drop_number=1
)

test_all   = pd.concat([X_test_id, X_ood], axis=0)
true_label = np.concatenate([
    np.ones(len(X_test_id)),  
    np.zeros(len(X_ood))      
])

print(f"\n Final test set: {len(test_all)} cells")
print(f"  - Other Cell types: {len(X_test_id)} (label=0)")
print(f"  - Least abundant cell type: {len(X_ood)} (label=1)")

In [ ]:
print("\n" + "="*70)
print("Step 7: Training attention model + Mahalanobis+Energy scoring")
print("="*70)

import sys
sys.path.insert(0, '/home')
import scMORCEL as sr

import time

total_start_time = time.time()

test_score, history = sr.scMORCEL(
    test=test_all,
    reference=X_train,
    label=y_train,
    processing_unit='cuda',
    max_epochs=150,
    patience=25,
    model_type='attention',
    #model_type='basic',
    attention_heads=4,
    score_function='mahalanobis_energy',
    mahal_energy_alpha=0.7,
    energy_temperature=1.0,
    use_validation=True,
    validation_split=0.2,
    learning_rate=5e-4,
    use_contrastive=True,
    contrastive_weight=0.03,
    contrastive_temperature=0.6,
    contrastive_type='contrastive',
    verbose=True
)

total_end_time = time.time()

total_seconds = total_end_time - total_start_time

In [ ]:
print("\n" + "="*70)
print("Step 8: Evaluating rare cell detection performance")
print("="*70)

from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, roc_curve)

auroc = roc_auc_score(true_label, test_score)
aupr  = average_precision_score(true_label, test_score)


fpr, tpr, thresholds = roc_curve(true_label, test_score)
idx   = np.argmin(np.abs(tpr - 0.95))
fpr95 = fpr[idx]

print(f"\nAUROC:   {auroc:.4f}")
print(f"AUPR:    {aupr:.4f}")
print(f"FPR95:   {fpr95:.4f}")
print(f"Total time: {total_seconds:.2f} seconds")